# 02 -- Exploratory Data Analysis (EDA)
## SME AutoFlow

**Author:** Hammad Ali (FA23-BCS-007)  
**Project:** SME AutoFlow -- GDGoC AI/ML Fellowship Final Project

---

This notebook explores `data/labeled_dataset.csv` before model training.  
We analyse class distributions, text characteristics, and node usage patterns to inform modelling decisions.

In [1]:
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from wordcloud import WordCloud

# Make matplotlib use a non-interactive backend for notebook execution
matplotlib.use('Agg')

sns.set_theme(style='whitegrid')
FIGSIZE = (10, 5)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

df = pd.read_csv(DATA_DIR / 'labeled_dataset.csv')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')

Loaded: 1000 rows x 3 columns


## Analysis 1 -- Dataset Overview

Basic metadata: shape, data types, and null value counts.

In [2]:
print('=== Shape ===')
print(f'{df.shape[0]} rows, {df.shape[1]} columns\n')

print('=== Data Types ===')
print(df.dtypes)

print('\n=== Null Value Counts ===')
print(df.isnull().sum())

print('\n=== Sample Row ===')
df.head()

=== Shape ===
1000 rows, 3 columns

=== Data Types ===
description    str
nodes          str
intent         str
dtype: object

=== Null Value Counts ===
description    0
nodes          6
intent         0
dtype: int64

=== Sample Row ===


,description,nodes,intent
0,💥 Viral TikTok Video Machine: Auto-Create Vide...,"n8n-nodes-base.googleSheets,n8n-nodes-base.htt...",team_communication
1,🎬 TikTok Video Downloader (No Watermark) - Tel...,"n8n-nodes-base.httpRequest,n8n-nodes-base.tele...",team_communication
2,Shopify Digital Product Automation\n(from just...,"n8n-nodes-base.airtable,n8n-nodes-base.httpReq...",data_sync
3,🧠 About this workflow\nThis workflow automatic...,"n8n-nodes-base.googleSheets,n8n-nodes-base.htt...",data_sync
4,PDF Invoice Extractor (AI)\n\nEnd-to-end pipel...,"n8n-nodes-base.httpRequest,n8n-nodes-base.goog...",data_sync


## Analysis 2 -- Intent Category Distribution

How many templates belong to each intent category?  
This reveals class imbalance that must be handled during training.

In [3]:
intent_counts = df['intent'].value_counts()

fig, ax = plt.subplots(figsize=FIGSIZE)
colors = sns.color_palette('viridis', n_colors=len(intent_counts))
bars = ax.barh(intent_counts.index[::-1], intent_counts.values[::-1], color=colors)

for bar, count in zip(bars, intent_counts.values[::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2,
            str(count), va='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Number of Templates', fontsize=12)
ax.set_ylabel('Intent Category', fontsize=12)
ax.set_title('Intent Category Distribution', fontsize=14, fontweight='bold')
ax.set_xlim(0, intent_counts.max() * 1.15)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'docs' / 'eda_intent_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

print(intent_counts.to_string())
print(f'\nImbalance ratio (max/min): {intent_counts.max()/intent_counts.min():.1f}x')

intent
data_sync             327
email_automation      254
team_communication    208
ai_tasks              119
general                60
database_ops           25
social_media            4
productivity            2
ecommerce               1

Imbalance ratio (max/min): 327.0x


C:\Users\User\AppData\Local\Temp\ipykernel_14008\929811476.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis 3 -- Description Length Distribution

How long are the template descriptions in words?  
Longer descriptions give the TF-IDF vectoriser more signal to work with.

In [4]:
df['word_count'] = df['description'].fillna('').apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.hist(df['word_count'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df['word_count'].median(), color='red', linestyle='--', linewidth=1.5,
           label=f'Median: {df["word_count"].median():.0f} words')
ax.axvline(df['word_count'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Mean: {df["word_count"].mean():.0f} words')
ax.set_xlabel('Word Count per Description', fontsize=12)
ax.set_ylabel('Number of Templates', fontsize=12)
ax.set_title('Description Length Distribution (Word Count)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(df['word_count'].describe().round(1))

count    1000.0
mean      439.2
std       269.6
min        44.0
25%       262.8
50%       374.5
75%       534.8
max      2289.0
Name: word_count, dtype: float64


C:\Users\User\AppData\Local\Temp\ipykernel_14008\577720906.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis 4 -- Top 20 Most Common n8n Nodes

Which n8n node types appear most frequently across all templates?  
These are the labels our node recommender must predict most reliably.

In [5]:
all_nodes = [
    n.strip()
    for nodes in df['nodes'].fillna('')
    for n in str(nodes).split(',')
    if n.strip()
]
node_freq = Counter(all_nodes).most_common(20)
names, counts = zip(*node_freq)
short_names = [n.split('.')[-1] for n in names]

fig, ax = plt.subplots(figsize=FIGSIZE)
colors = sns.color_palette('viridis', n_colors=20)
bars = ax.barh(short_names[::-1], list(counts)[::-1], color=colors)
for bar, c in zip(bars, list(counts)[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            str(c), va='center', fontsize=9)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Most Common n8n Node Types', fontsize=14, fontweight='bold')
ax.set_xlim(0, max(counts) * 1.12)
plt.tight_layout()
plt.show()

print(f'Total unique node types: {len(set(all_nodes))}')

Total unique node types: 151


C:\Users\User\AppData\Local\Temp\ipykernel_14008\1696508576.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis 5 -- Average Description Length per Intent Category

Do some intent categories have longer descriptions than others?  
This can indicate how information-dense each category tends to be.

In [6]:
avg_len = df.groupby('intent')['word_count'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=FIGSIZE)
bars = ax.bar(avg_len.index, avg_len.values,
              color=sns.color_palette('viridis', n_colors=len(avg_len)))
for bar, val in zip(bars, avg_len.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{val:.0f}', ha='center', fontweight='bold', fontsize=10)
ax.set_xticklabels(avg_len.index, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Average Word Count', fontsize=12)
ax.set_title('Average Description Length per Intent Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(avg_len.round(1).to_string())

C:\Users\User\AppData\Local\Temp\ipykernel_14008\2193322099.py:9: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(avg_len.index, rotation=35, ha='right', fontsize=10)
C:\Users\User\AppData\Local\Temp\ipykernel_14008\2193322099.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


intent
ecommerce             458.0
email_automation      451.8
data_sync             442.6
ai_tasks              435.8
team_communication    434.6
general               430.0
database_ops          389.1
productivity          342.0
social_media          196.8


## Analysis 6 -- Word Cloud of Descriptions

A visual representation of the most frequent words across all template descriptions.  
This gives an intuitive sense of the vocabulary used in the dataset.

In [7]:
all_text = ' '.join(df['description'].fillna('').astype(str).tolist())

# Common stopwords that aren't useful for visualisation
extra_stops = {'workflow', 'n8n', 'node', 'use', 'using', 'used',
               'can', 'will', 'also', 'make', 'get', 'set', 'new',
               'one', 'step', 'allow', 'need', 'want', 'work', 'well'}

wc = WordCloud(
    width=1000, height=500,
    background_color='white',
    stopwords=WordCloud().stopwords | extra_stops,
    colormap='viridis',
    max_words=150,
    collocations=False,
).generate(all_text)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud of Template Descriptions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'docs' / 'eda_wordcloud.png'), dpi=150, bbox_inches='tight')
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_14008\1345797449.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary -- Key EDA Findings

1. **Severe class imbalance**: `data_sync` (327) and `email_automation` (254) dominate the dataset, while `ecommerce` (1), `productivity` (2), and `social_media` (4) are extremely rare. We will use `class_weight='balanced'` in the classifiers and merge classes with < 5 samples into `general`.

2. **Description length varies widely**: Word counts range from ~10 to 2000+, with a median of ~200 words. TF-IDF with `sublinear_tf=True` normalises this variation effectively.

3. **httpRequest and code nodes dominate**: The generic `httpRequest` and `code` nodes appear in the majority of templates, which may reduce the node recommender's discriminative power. Rare nodes (< 5 occurrences) are filtered out.

4. **`ai_tasks` descriptions are longest**: AI-related workflows tend to have more detailed descriptions, which may give the classifier a strong signal.

5. **Vocabulary is domain-specific**: The word cloud reveals n8n-specific terms (data, send, google, email, slack, trigger) that TF-IDF will capture well, but generic English stopwords are abundant and require proper filtering.